# OSS Health Feature Engineering and Modeling

이 노트북은 기존 feature family를 기준으로 다음 흐름을 고정한다.

1. 원본 `label` 예측 실험을 메인 실험으로 사용한다.
2. `new_label` 예측 실험은 health score 기반 pseudo-label 검증으로 분리한다.
3. 두 실험 모두 동일한 최종 feature set을 사용한다.
4. `oss_health_score`와 dimension score는 leakage 컬럼으로 간주하고 모델 feature에서 제외한다.
5. 선형 분석, 비선형 분석, ablation test, classification report를 target별로 따로 확인한다.


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from scipy.stats import mannwhitneyu

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

RANDOM_STATE = 42


In [2]:
DATA_PATH = Path("data/main/dataset.csv")

if not DATA_PATH.exists():
    DATA_PATH = Path("/Users/carolyn/Desktop/3-1/opensource/data/oss-health-data/src/data/main/dataset.csv")

df = pd.read_csv(DATA_PATH)

print("Data shape:", df.shape)
print("Label distribution:")
print(df["label"].value_counts())

df.head()


Data shape: (411, 83)
Label distribution:
label
1    274
0    137
Name: count, dtype: int64


,Unnamed: 0,repo_name,num_contributors,total_contributions,top1_contribution_share,top5_contribution_share,contribution_gini,median_contributions,num_deployments,has_deployments,num_unique_refs,tag_based_deployment_ratio,deployment_recency_days,num_events,num_unique_event_types,dominant_event_type,dominant_event_ratio,event_type_entropy,has_IssuesEvent,has_PullRequestEvent,has_IssueCommentEvent,recent_event_density,IssuesEvent_ratio,IssueCommentEvent_ratio,PullRequestEvent_ratio,PushEvent_ratio,WatchEvent_ratio,ForkEvent_ratio,stargazers_count,subscribers_count,subscribers_to_stars_ratio,primary_language_ratio,top2_ratio,top3_ratio,language_entropy,minor_lang_ratio,infra_ratio,markup_ratio,is_monolingual,compiled_ratio,num_tags,stable_tag_ratio,prerelease_tag_ratio,latest_tag_is_stable,latest_tag_is_prerelease,semver_tag_ratio,num_major_versions,num_minor_versions,repo_age_days,last_update_recency_days,last_push_recency_days,update_push_gap_days,repo_size,forks_count,open_issues_count,network_count,has_issues,has_projects,has_downloads,has_wiki,has_pages,has_discussions,archived,disabled,allow_forking,has_pull_requests,forks_to_stars_ratio,open_issues_to_stars_ratio,stars_per_repo_age_day,forks_per_repo_age_day,top1_contribution_ratio,top3_contribution_ratio,contribution_entropy,contributors_to_stars_ratio,tag_release_velocity,interaction_ratio,development_ratio,external_interest_event_ratio,stars_per_size,forks_per_size,issues_per_size,error,label
0,0,feast-dev/feast,30.0,3139.0,0.173941,0.475948,0.475385,61.5,30.0,1.0,2.0,0.0,24.292122,30.0,8.0,PushEvent,0.366667,1.814285,0.0,1.0,1.0,14.478587,0.000000,0.100000,0.100000,0.366667,0.166667,0.100000,7004.0,85.0,0.012136,0.763338,0.889846,0.939283,0.894262,0.236662,0.012359,0.003585,0.0,0.153934,30.0,1.000000,0.000000,1.0,0.0,1.000000,1.0,24.0,2703.643302,1.189633,1.457353,0.267720,254431.0,1315.0,346.0,1315.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.187750,0.049400,2.590578,0.486381,0.173941,0.339599,3.005782,0.004283,0.011096,0.100000,0.466667,0.266667,0.027528,0.005168,0.001360,NaN,1
1,1,tmux/tmux,19.0,9800.0,0.802143,0.997551,0.923963,2.0,0.0,0.0,0.0,0.0,0.000000,30.0,6.0,IssueCommentEvent,0.466667,1.244199,1.0,1.0,1.0,24.062831,0.033333,0.466667,0.033333,0.033333,0.366667,0.000000,45140.0,470.0,0.010412,0.872271,0.944649,0.972869,0.541481,0.127729,0.030302,0.000000,0.0,0.872271,30.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,3988.970992,0.056640,0.324546,0.267905,19142.0,2603.0,55.0,2603.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.057665,0.001218,11.316202,0.652549,0.802143,0.996735,0.529017,0.000421,0.007521,0.500000,0.066667,0.366667,2.358165,0.135984,0.002873,NaN,1
2,2,ultralytics/ultralytics,30.0,3551.0,0.443537,0.813292,0.779283,21.5,30.0,1.0,1.0,0.0,0.788279,30.0,7.0,IssueCommentEvent,0.333333,1.636432,0.0,1.0,1.0,12.361163,0.000000,0.333333,0.133333,0.233333,0.200000,0.033333,57179.0,256.0,0.004477,0.996246,0.997740,0.999058,0.028770,0.003754,0.002435,0.001319,1.0,0.000000,30.0,1.000000,0.000000,1.0,0.0,1.000000,1.0,1.0,1342.516161,0.064101,0.082318,0.018218,55086.0,11006.0,325.0,11006.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.192483,0.005684,42.590921,8.198039,0.443537,0.666291,2.058244,0.000525,0.022346,0.333333,0.366667,0.233333,1.037995,0.199797,0.005900,NaN,1
3,3,denoland/fresh,30.0,1396.0,0.474928,0.803009,0.767240,8.5,30.0,1.0,30.0,0.0,177.442465,30.0,7.0,IssueCommentEvent,0.233333,1.869789,1.0,1.0,1.0,15.180977,0.100000,0.233333,0.166667,0.200000,0.133333,0.000000,13757.0,78.0,0.005670,0.965551,0.999287,0.999713,0.153832,0.034449,0.000000,0.034162,1.0,0.000000,30.0,1.000000,0.000000,1.0,0.0,1.000000,2.0,7.0,1834.599769,0.015093,0.613854,0.598762,45712.0,749.0,82.0,749.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.054445,0.005961,7.498638,0.408263,0.474928,0.704155,2.023641,0.002181,0.016352,0.333333,0.366667,0.133333,0.300949,0.016385,0.001794,NaN,1
4,4,milvus-io/milvus,30.0,17783.0,0.096947,0.307372,0.252978,506.5,0.0,0.0,0.0,0.0,0.000000,30.0,5.0,PullReque

## Feature Family 정의

In [3]:
feature_families = {
    "contributor": [
        "num_contributors",
        "total_contributions",
        "top1_contribution_share",
        "top5_contribution_share",
        "contribution_gini",
        "median_contributions",
        "top1_contribution_ratio",
        "top3_contribution_ratio",
        "contribution_entropy",
        "contributors_to_stars_ratio",
    ],
    "release": [
        "num_deployments",
        "has_deployments",
        "num_unique_refs",
        "tag_based_deployment_ratio",
        "deployment_recency_days",
        "num_tags",
        "stable_tag_ratio",
        "prerelease_tag_ratio",
        "latest_tag_is_stable",
        "latest_tag_is_prerelease",
        "semver_tag_ratio",
        "num_major_versions",
        "num_minor_versions",
        "tag_release_velocity",
    ],
    "event": [
        "num_events",
        "num_unique_event_types",
        "dominant_event_ratio",
        "event_type_entropy",
        "has_IssuesEvent",
        "has_PullRequestEvent",
        "has_IssueCommentEvent",
        "recent_event_density",
        "IssuesEvent_ratio",
        "IssueCommentEvent_ratio",
        "PullRequestEvent_ratio",
        "PushEvent_ratio",
        "WatchEvent_ratio",
        "ForkEvent_ratio",
        "interaction_ratio",
        "development_ratio",
        "external_interest_event_ratio",
    ],
    "popularity": [
        "stargazers_count",
        "subscribers_count",
        "subscribers_to_stars_ratio",
        "forks_count",
        "network_count",
        "forks_to_stars_ratio",
        "open_issues_to_stars_ratio",
        "stars_per_repo_age_day",
        "forks_per_repo_age_day",
        "stars_per_size",
        "forks_per_size",
    ],
    "language": [
        "primary_language_ratio",
        "top2_ratio",
        "top3_ratio",
        "language_entropy",
        "minor_lang_ratio",
        "infra_ratio",
        "markup_ratio",
        "is_monolingual",
        "compiled_ratio",
    ],
    "maintenance": [
        "repo_age_days",
        "last_update_recency_days",
        "last_push_recency_days",
        "update_push_gap_days",
    ],
    "governance": [
        "repo_size",
        "open_issues_count",
        "has_issues",
        "has_projects",
        "has_downloads",
        "has_wiki",
        "has_pages",
        "has_discussions",
        "archived",
        "disabled",
        "allow_forking",
        "has_pull_requests",
        "issues_per_size",
    ],
}

negative_features = [
    "top1_contribution_share",
    "top5_contribution_share",
    "contribution_gini",
    "top1_contribution_ratio",
    "top3_contribution_ratio",
    "deployment_recency_days",
    "prerelease_tag_ratio",
    "latest_tag_is_prerelease",
    "last_update_recency_days",
    "last_push_recency_days",
    "update_push_gap_days",
    "open_issues_to_stars_ratio",
    "issues_per_size",
    "dominant_event_ratio",
    "archived",
    "disabled",
]

score_dimensions = {
    "community_activity": [
        "num_events",
        "num_unique_event_types",
        "event_type_entropy",
        "recent_event_density",
        "has_IssuesEvent",
        "has_PullRequestEvent",
        "has_IssueCommentEvent",
        "IssuesEvent_ratio",
        "IssueCommentEvent_ratio",
        "PullRequestEvent_ratio",
        "PushEvent_ratio",
        "interaction_ratio",
        "development_ratio",
    ],
    "contributor_sustainability": feature_families["contributor"],
    "release_engineering": feature_families["release"],
    "popularity_adoption": [
        "stargazers_count",
        "subscribers_count",
        "forks_count",
        "network_count",
        "subscribers_to_stars_ratio",
        "forks_to_stars_ratio",
        "stars_per_repo_age_day",
        "forks_per_repo_age_day",
        "stars_per_size",
        "forks_per_size",
        "external_interest_event_ratio",
    ],
    "language_structure": feature_families["language"],
    "maintenance": feature_families["maintenance"],
    "governance": [
        "repo_size",
        "open_issues_count",
        "has_issues",
        "has_projects",
        "has_downloads",
        "has_wiki",
        "has_pages",
        "has_discussions",
        "archived",
        "disabled",
        "allow_forking",
        "has_pull_requests",
        "open_issues_to_stars_ratio",
        "issues_per_size",
    ],
}


## Health Score 기반 Pseudo-label 생성

`new_label`은 원본 정답이 아니라 feature 기반 score로 만든 pseudo-label이다. 따라서 모델의 메인 target은 `label`로 두고, `new_label`은 보조 검증 target으로만 사용한다.

In [4]:
def existing_columns(columns, data):
    return [col for col in columns if col in data.columns]


def numeric_frame(data, columns):
    out = data[columns].copy()

    for col in columns:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    return out


def build_health_score_labels(data):
    out = data.copy()
    dimensions = {
        dim: existing_columns(cols, out)
        for dim, cols in score_dimensions.items()
    }

    all_score_features = sorted(set(sum(dimensions.values(), [])))
    numeric = numeric_frame(out, all_score_features)

    for col in all_score_features:
        numeric[col] = numeric[col].fillna(numeric[col].median())

    scaled = pd.DataFrame(index=out.index)

    for col in all_score_features:
        scaled[col] = MinMaxScaler().fit_transform(numeric[[col]]).ravel()

        if col in negative_features:
            scaled[col] = 1 - scaled[col]

    for dim, cols in dimensions.items():
        if cols:
            out[f"{dim}_score"] = scaled[cols].mean(axis=1)
        else:
            out[f"{dim}_score"] = np.nan

    dimension_score_cols = [
        f"{dim}_score"
        for dim in dimensions
    ]

    out["oss_health_score"] = out[dimension_score_cols].mean(axis=1)
    threshold = out["oss_health_score"].median()
    out["new_label"] = (out["oss_health_score"] >= threshold).astype(int)

    return out, dimension_score_cols


df_labeled, dimension_score_cols = build_health_score_labels(df)

print("Original label distribution:")
print(df_labeled["label"].value_counts())
print()
print("Pseudo-label distribution:")
print(df_labeled["new_label"].value_counts())
print()
print("Original label vs pseudo-label:")
print(pd.crosstab(df_labeled["label"], df_labeled["new_label"]))
print()
print("Agreement:", (df_labeled["label"] == df_labeled["new_label"]).mean())


Original label distribution:
label
1    274
0    137
Name: count, dtype: int64

Pseudo-label distribution:
new_label
1    206
0    205
Name: count, dtype: int64

Original label vs pseudo-label:
new_label    0    1
label              
0          111   26
1           94  180

Agreement: 0.708029197080292


## Feature Engineering

In [5]:
def add_engineered_features(data):
    out = data.copy()
    eps = 1e-9
    source_cols = sorted(set(sum(feature_families.values(), [])))

    for col in existing_columns(source_cols, out):
        out[col] = pd.to_numeric(out[col], errors="coerce")

    out["is_recently_pushed_30d"] = (
        out["last_push_recency_days"] <= 30
    ).astype(int)

    out["is_recently_updated_90d"] = (
        out["last_update_recency_days"] <= 90
    ).astype(int)

    out["push_update_consistency"] = (
        1 / (1 + out["update_push_gap_days"])
    )

    out["bus_factor_risk"] = (
        out["top1_contribution_share"]
        + out["contribution_gini"]
    ) / 2

    out["distributed_contribution_score"] = (
        out["contribution_entropy"]
        * (1 - out["top1_contribution_share"])
    )

    out["release_maturity_score"] = (
        out["stable_tag_ratio"]
        * out["semver_tag_ratio"]
        * out["latest_tag_is_stable"]
    )

    out["active_release_score"] = (
        np.log1p(out["num_tags"])
        * out["tag_release_velocity"]
    )

    out["collaboration_event_score"] = (
        out["PullRequestEvent_ratio"]
        + out["IssueCommentEvent_ratio"]
        + out["IssuesEvent_ratio"]
    )

    out["activity_diversity_score"] = (
        out["event_type_entropy"]
        * out["num_unique_event_types"]
    )

    out["adoption_efficiency"] = (
        np.log1p(out["stargazers_count"])
        / np.log1p(out["repo_age_days"] + 1)
    )

    out["issue_burden_score"] = (
        out["open_issues_count"]
        / (np.log1p(out["stargazers_count"]) + eps)
    )

    out["governance_openness_score"] = (
        out["has_issues"]
        + out["has_projects"]
        + out["has_wiki"]
        + out["has_discussions"]
        + out["has_pull_requests"]
    ) / 5

    out["release_recency_score"] = (
        1 / (1 + out["deployment_recency_days"])
    )

    out["maintainer_activity_score"] = (
        out["is_recently_pushed_30d"]
        + out["is_recently_updated_90d"]
        + out["push_update_consistency"]
    ) / 3

    out["healthy_activity_score"] = (
        out["interaction_ratio"]
        + out["development_ratio"]
        + out["collaboration_event_score"]
    ) / 3

    out.replace([np.inf, -np.inf], np.nan, inplace=True)

    return out


df_fe = add_engineered_features(df_labeled)


## 최종 Feature Set

In [6]:
keep_raw_features = [
    "last_push_recency_days",
    "last_update_recency_days",
    "update_push_gap_days",
    "repo_age_days",
    "event_type_entropy",
    "num_unique_event_types",
    "num_events",
    "has_IssueCommentEvent",
    "has_PullRequestEvent",
    "interaction_ratio",
    "development_ratio",
    "IssueCommentEvent_ratio",
    "PullRequestEvent_ratio",
    "external_interest_event_ratio",
    "num_tags",
    "tag_release_velocity",
    "semver_tag_ratio",
    "stable_tag_ratio",
    "latest_tag_is_stable",
    "deployment_recency_days",
    "has_discussions",
    "archived",
    "disabled",
    "has_issues",
    "has_pull_requests",
    "total_contributions",
    "median_contributions",
    "contribution_entropy",
    "top1_contribution_share",
    "contribution_gini",
    "top5_contribution_share",
    "subscribers_to_stars_ratio",
    "forks_to_stars_ratio",
    "open_issues_to_stars_ratio",
    "compiled_ratio",
    "language_entropy",
]

engineered_features = [
    "is_recently_pushed_30d",
    "is_recently_updated_90d",
    "push_update_consistency",
    "bus_factor_risk",
    "distributed_contribution_score",
    "release_maturity_score",
    "active_release_score",
    "collaboration_event_score",
    "activity_diversity_score",
    "adoption_efficiency",
    "issue_burden_score",
    "governance_openness_score",
    "release_recency_score",
    "maintainer_activity_score",
    "healthy_activity_score",
]

rejected_features = [
    "Unnamed: 0",
    "repo_name",
    "dominant_event_type",
    "error",
    "top2_ratio",
    "top3_ratio",
    "primary_language_ratio",
    "minor_lang_ratio",
    "markup_ratio",
    "is_monolingual",
    "has_downloads",
    "has_pages",
    "has_projects",
    "has_wiki",
    "allow_forking",
    "num_deployments",
    "has_deployments",
    "num_unique_refs",
    "tag_based_deployment_ratio",
    "latest_tag_is_prerelease",
    "prerelease_tag_ratio",
    "stargazers_count",
    "forks_count",
    "network_count",
    "subscribers_count",
    "repo_size",
    "open_issues_count",
    "top1_contribution_ratio",
    "top3_contribution_ratio",
    "contributors_to_stars_ratio",
    "dominant_event_ratio",
    "WatchEvent_ratio",
    "ForkEvent_ratio",
    "PushEvent_ratio",
    "recent_event_density",
]

leakage_columns = ["oss_health_score"] + dimension_score_cols

final_features = existing_columns(
    keep_raw_features + engineered_features,
    df_fe,
)

for col in leakage_columns + ["label", "new_label", "repo_name", "dominant_event_type", "error"]:
    if col in final_features:
        raise ValueError(f"Leakage or non-feature column included: {col}")

modeling_df = df_fe[final_features + ["label", "new_label"]].copy()

for col in final_features:
    modeling_df[col] = pd.to_numeric(modeling_df[col], errors="coerce")

print("Final modeling dataset shape:", modeling_df.shape)
print("Number of final features:", len(final_features))


Final modeling dataset shape: (411, 53)
Number of final features: 51


In [7]:
feature_family_map = {
    "maintenance": [
        "last_push_recency_days",
        "last_update_recency_days",
        "update_push_gap_days",
        "repo_age_days",
        "is_recently_pushed_30d",
        "is_recently_updated_90d",
        "push_update_consistency",
        "maintainer_activity_score",
    ],
    "event": [
        "event_type_entropy",
        "num_unique_event_types",
        "num_events",
        "has_IssueCommentEvent",
        "has_PullRequestEvent",
        "interaction_ratio",
        "development_ratio",
        "IssueCommentEvent_ratio",
        "PullRequestEvent_ratio",
        "external_interest_event_ratio",
        "collaboration_event_score",
        "activity_diversity_score",
        "healthy_activity_score",
    ],
    "release": [
        "num_tags",
        "tag_release_velocity",
        "semver_tag_ratio",
        "stable_tag_ratio",
        "latest_tag_is_stable",
        "deployment_recency_days",
        "release_maturity_score",
        "active_release_score",
        "release_recency_score",
    ],
    "governance": [
        "has_discussions",
        "archived",
        "disabled",
        "has_issues",
        "has_pull_requests",
        "governance_openness_score",
    ],
    "contributor": [
        "total_contributions",
        "median_contributions",
        "contribution_entropy",
        "top1_contribution_share",
        "contribution_gini",
        "top5_contribution_share",
        "bus_factor_risk",
        "distributed_contribution_score",
    ],
    "popularity_adjusted": [
        "subscribers_to_stars_ratio",
        "forks_to_stars_ratio",
        "open_issues_to_stars_ratio",
        "adoption_efficiency",
        "issue_burden_score",
    ],
    "language": [
        "compiled_ratio",
        "language_entropy",
    ],
}

feature_to_family = {
    col: family
    for family, cols in feature_family_map.items()
    for col in cols
}

feature_selection_table = pd.DataFrame({
    "feature": final_features,
    "family": [feature_to_family.get(col, "other") for col in final_features],
    "source": ["engineered" if col in engineered_features else "raw" for col in final_features],
})

rejected_feature_table = pd.DataFrame({
    "feature": existing_columns(rejected_features, df_fe),
})

feature_selection_table


,feature,family,source
0,last_push_recency_days,maintenance,raw
1,last_update_recency_days,maintenance,raw
2,update_push_gap_days,maintenance,raw
3,repo_age_days,maintenance,raw
4,event_type_entropy,event,raw
5,num_unique_event_types,event,raw
6,num_events,event,raw
7,has_IssueCommentEvent,event,raw
8,has_PullRequestEvent,event,raw
9,interaction_ratio,event,raw


## 공통 분석 함수

In [8]:
def get_xy(data, target_col):
    X = data[final_features].copy()
    y = data[target_col].astype(int)

    return X, y


def get_models():
    return {
        "LogisticRegression": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=20000,
                solver="liblinear",
                class_weight="balanced",
                random_state=RANDOM_STATE,
            )),
        ]),
        "RandomForest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=500,
                min_samples_leaf=2,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=1,
            )),
        ]),
        "ExtraTrees": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesClassifier(
                n_estimators=700,
                min_samples_leaf=2,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=1,
            )),
        ]),
        "GradientBoosting": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", GradientBoostingClassifier(
                random_state=RANDOM_STATE,
            )),
        ]),
    }


def evaluate_models(data, target_col, test_size=0.2):
    X, y = get_xy(data, target_col)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    rows = []
    reports = {}
    fitted_models = {}

    print("X_train:", X_train.shape)
    print("X_test :", X_test.shape)
    print("y_train distribution:")
    print(y_train.value_counts(normalize=True))

    for model_name, model in get_models().items():
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        rows.append({
            "target": target_col,
            "model": model_name,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, y_prob),
        })

        reports[model_name] = {
            "classification_report": classification_report(y_test, y_pred),
            "confusion_matrix": confusion_matrix(y_test, y_pred),
        }

        fitted_models[model_name] = model

        print()
        print("=" * 70)
        print(model_name)
        print("=" * 70)
        print(classification_report(y_test, y_pred))
        print("Confusion Matrix")
        print(confusion_matrix(y_test, y_pred))

    result = (
        pd.DataFrame(rows)
        .sort_values("roc_auc", ascending=False)
        .reset_index(drop=True)
    )

    return result, reports, fitted_models, (X_train, X_test, y_train, y_test)


In [9]:
def univariate_analysis(data, target_col):
    rows = []

    for feature in final_features:
        unhealthy = data.loc[data[target_col] == 0, feature].dropna()
        healthy = data.loc[data[target_col] == 1, feature].dropna()

        unhealthy_mean = unhealthy.mean()
        healthy_mean = healthy.mean()
        unhealthy_median = unhealthy.median()
        healthy_median = healthy.median()

        pooled_std = np.sqrt(
            (unhealthy.var(ddof=1) + healthy.var(ddof=1)) / 2
        )

        cohen_d = (
            (healthy_mean - unhealthy_mean) / pooled_std
            if pooled_std > 0
            else np.nan
        )

        try:
            p_value = mannwhitneyu(
                unhealthy,
                healthy,
                alternative="two-sided",
            ).pvalue
        except ValueError:
            p_value = np.nan

        rows.append({
            "target": target_col,
            "feature": feature,
            "family": feature_to_family.get(feature, "other"),
            "source": "engineered" if feature in engineered_features else "raw",
            "unhealthy_mean": unhealthy_mean,
            "healthy_mean": healthy_mean,
            "unhealthy_median": unhealthy_median,
            "healthy_median": healthy_median,
            "mean_diff_healthy_minus_unhealthy": healthy_mean - unhealthy_mean,
            "median_diff_healthy_minus_unhealthy": healthy_median - unhealthy_median,
            "cohen_d": cohen_d,
            "abs_cohen_d": abs(cohen_d),
            "p_value": p_value,
        })

    return (
        pd.DataFrame(rows)
        .sort_values(["abs_cohen_d", "p_value"], ascending=[False, True])
        .reset_index(drop=True)
    )


def linear_odds_ratio(data, target_col):
    X, y = get_xy(data, target_col)

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=20000,
            solver="liblinear",
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ])

    model.fit(X, y)
    coef = model.named_steps["model"].coef_[0]

    return (
        pd.DataFrame({
            "target": target_col,
            "feature": final_features,
            "family": [feature_to_family.get(col, "other") for col in final_features],
            "coef": coef,
            "odds_ratio_per_1sd": np.exp(coef),
            "abs_coef": np.abs(coef),
        })
        .sort_values("abs_coef", ascending=False)
        .reset_index(drop=True)
    )


def nonlinear_importance(data, target_col):
    X, y = get_xy(data, target_col)

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesClassifier(
            n_estimators=700,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=1,
        )),
    ])

    model.fit(X, y)
    importance = model.named_steps["model"].feature_importances_

    return (
        pd.DataFrame({
            "target": target_col,
            "feature": final_features,
            "family": [feature_to_family.get(col, "other") for col in final_features],
            "importance": importance,
        })
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )


def ablation_test(data, target_col):
    X, y = get_xy(data, target_col)

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesClassifier(
            n_estimators=500,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=1,
        )),
    ])

    rows = []

    all_scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="roc_auc",
        n_jobs=1,
    )

    rows.append({
        "target": target_col,
        "setting": "all_features",
        "roc_auc_mean": all_scores.mean(),
        "roc_auc_std": all_scores.std(),
        "num_features": X.shape[1],
    })

    families = sorted(
        set(feature_to_family[col] for col in final_features if col in feature_to_family)
    )

    for family in families:
        family_cols = [
            col for col in final_features
            if feature_to_family.get(col) == family
        ]

        scores = cross_val_score(
            model,
            X[family_cols],
            y,
            cv=cv,
            scoring="roc_auc",
            n_jobs=1,
        )

        rows.append({
            "target": target_col,
            "setting": f"only_{family}",
            "roc_auc_mean": scores.mean(),
            "roc_auc_std": scores.std(),
            "num_features": len(family_cols),
        })

    for family in families:
        remaining_cols = [
            col for col in final_features
            if feature_to_family.get(col) != family
        ]

        scores = cross_val_score(
            model,
            X[remaining_cols],
            y,
            cv=cv,
            scoring="roc_auc",
            n_jobs=1,
        )

        rows.append({
            "target": target_col,
            "setting": f"without_{family}",
            "roc_auc_mean": scores.mean(),
            "roc_auc_std": scores.std(),
            "num_features": len(remaining_cols),
        })

    return (
        pd.DataFrame(rows)
        .sort_values("roc_auc_mean", ascending=False)
        .reset_index(drop=True)
    )


## Main Experiment: Original Label

In [10]:
target_col = "label"

label_univariate = univariate_analysis(modeling_df, target_col)
label_linear_result = linear_odds_ratio(modeling_df, target_col)
label_nonlinear_result = nonlinear_importance(modeling_df, target_col)
label_ablation_result = ablation_test(modeling_df, target_col)

label_results, label_reports, label_models, label_split = evaluate_models(
    modeling_df,
    target_col,
)

print()
print("=" * 70)
print("FINAL MODEL PERFORMANCE: original label")
print("=" * 70)

label_results


X_train: (328, 51)
X_test : (83, 51)
y_train distribution:
label
1    0.667683
0    0.332317
Name: proportion, dtype: float64

LogisticRegression
              precision    recall  f1-score   support

           0       0.61      0.71      0.66        28
           1       0.84      0.76      0.80        55

    accuracy                           0.75        83
   macro avg       0.72      0.74      0.73        83
weighted avg       0.76      0.75      0.75        83

Confusion Matrix
[[20  8]
 [13 42]]

RandomForest
              precision    recall  f1-score   support

           0       0.52      0.54      0.53        28
           1       0.76      0.75      0.75        55

    accuracy                           0.67        83
   macro avg       0.64      0.64      0.64        83
weighted avg       0.68      0.67      0.68        83

Confusion Matrix
[[15 13]
 [14 41]]

ExtraTrees
              precision    recall  f1-score   support

           0       0.53      0.57      0.55    

,target,model,accuracy,precision,recall,f1,roc_auc
0,label,LogisticRegression,0.746988,0.840000,0.763636,0.800000,0.876299
1,label,GradientBoosting,0.698795,0.777778,0.763636,0.770642,0.796104
2,label,RandomForest,0.674699,0.759259,0.745455,0.752294,0.787662
3,label,ExtraTrees,0.686747,0.773585,0.745455,0.759259,0.768182


In [11]:
label_univariate.head(30)


,target,feature,family,source,unhealthy_mean,healthy_mean,unhealthy_median,healthy_median,mean_diff_healthy_minus_unhealthy,median_diff_healthy_minus_unhealthy,cohen_d,abs_cohen_d,p_value
0,label,push_update_consistency,maintenance,engineered,0.163965,0.599290,0.008874,0.771777,0.435326,0.762903,1.256009,1.256009,5.968249e-25
1,label,maintainer_activity_score,maintenance,engineered,0.502343,0.808832,0.336291,0.923926,0.306489,0.587634,1.240071,1.240071,3.758936e-25
2,label,event_type_entropy,event,raw,0.645274,1.320618,0.540204,1.499143,0.675344,0.958939,1.216863,1.216863,2.286700e-23
3,label,num_unique_event_types,event,raw,3.007299,5.702206,3.000000,6.000000,2.694907,3.000000,1.187925,1.187925,1.255405e-22
4,label,activity_diversity_score,event,engineered,3.211243,8.611394,1.298497,9.013406,5.400151,7.714909,1.134674,1.134674,3.969572e-23
5,label,num_events,event,raw,18.781022,28.441176,28.000000,30.000000,9.660155,2.000000,1.011952,1.011952,9.864619e-19
6,label,is_recently_pushed_30d,maintenance,engineered,0.379562,0.821168,0.000000,1.000000,0.441606,1.000000,1.007016,1.007016,2.433433e-19
7,label,has_IssueCommentEvent,event,raw,0.335766,0.783088,0.000000,1.000000,0.447322,1.000000,1.006355,1.006355,8.864402e-19
8,label,active_release_score,release,engineered,0.012534,0.027591,0.016815,0.025785,0.015057,0.008970,0.974320,0.974320,1.133026e-19
9,label,tag_release_velocity,release,raw,0.003751,0.008091,0.005046,0.007512,0.004341,0.002466,0.960884,0.960884,1.270150e-19


In [12]:
label_linear_result.head(30)


,target,feature,family,coef,odds_ratio_per_1sd,abs_coef
0,label,last_update_recency_days,maintenance,-0.847887,0.428319,0.847887
1,label,total_contributions,contributor,0.771726,2.163497,0.771726
2,label,num_unique_event_types,event,-0.745653,0.474424,0.745653
3,label,open_issues_to_stars_ratio,popularity_adjusted,0.702916,2.019634,0.702916
4,label,subscribers_to_stars_ratio,popularity_adjusted,-0.661154,0.516255,0.661154
5,label,activity_diversity_score,event,0.645765,1.907446,0.645765
6,label,tag_release_velocity,release,0.597722,1.817973,0.597722
7,label,event_type_entropy,event,0.580086,1.786192,0.580086
8,label,active_release_score,release,0.544400,1.723574,0.544400
9,label,compiled_ratio,language,0.538505,1.713443,0.538505


In [13]:
label_nonlinear_result.head(30)


,target,feature,family,importance
0,label,push_update_consistency,maintenance,0.087727
1,label,has_IssueCommentEvent,event,0.068002
2,label,maintainer_activity_score,maintenance,0.059249
3,label,num_events,event,0.048008
4,label,is_recently_pushed_30d,maintenance,0.045798
5,label,num_tags,release,0.042558
6,label,active_release_score,release,0.036503
7,label,tag_release_velocity,release,0.035646
8,label,event_type_entropy,event,0.034125
9,label,has_PullRequestEvent,event,0.033100


In [14]:
label_ablation_result


,target,setting,roc_auc_mean,roc_auc_std,num_features
0,label,only_maintenance,0.758042,0.029740,8
1,label,without_contributor,0.738283,0.063904,43
2,label,without_governance,0.737282,0.056366,45
3,label,all_features,0.725082,0.063750,51
4,label,without_maintenance,0.723658,0.069701,43
5,label,without_popularity_adjusted,0.723514,0.059816,46
6,label,without_event,0.722686,0.061732,38
7,label,without_language,0.720695,0.059016,49
8,label,only_event,0.714606,0.052086,13
9,label,without_release,0.710914,0.054724,42


## Optional SHAP Analysis

SHAP은 설치되어 있을 때만 실행한다. 설치되어 있지 않으면 linear odds ratio와 tree importance를 해석 근거로 사용한다.

In [15]:
try:
    import shap

    X_train, X_test, y_train, y_test = label_split
    best_tree_model = label_models["ExtraTrees"]
    transformed_X_test = best_tree_model.named_steps["imputer"].transform(X_test)
    tree_model = best_tree_model.named_steps["model"]

    explainer = shap.TreeExplainer(tree_model)
    shap_values = explainer.shap_values(transformed_X_test)

    if isinstance(shap_values, list):
        shap_for_positive_class = np.asarray(shap_values[1])
    else:
        shap_array = np.asarray(shap_values)

        if shap_array.ndim == 3 and shap_array.shape[2] == 2:
            shap_for_positive_class = shap_array[:, :, 1]
        elif shap_array.ndim == 3 and shap_array.shape[0] == 2:
            shap_for_positive_class = shap_array[1, :, :]
        elif shap_array.ndim == 2:
            shap_for_positive_class = shap_array
        else:
            raise ValueError(f"Unexpected SHAP value shape: {shap_array.shape}")

    mean_abs_shap = np.abs(shap_for_positive_class).mean(axis=0).ravel()

    shap_importance = pd.DataFrame({
        "feature": final_features,
        "family": [feature_to_family.get(col, "other") for col in final_features],
        "mean_abs_shap": mean_abs_shap,
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

except ImportError:
    shap_importance = pd.DataFrame()
    print("SHAP is not installed.")

shap_importance.head(30)


,feature,family,mean_abs_shap
0,push_update_consistency,maintenance,0.040175
1,has_IssueCommentEvent,event,0.037430
2,maintainer_activity_score,maintenance,0.027067
3,num_events,event,0.025210
4,is_recently_pushed_30d,maintenance,0.024083
5,num_tags,release,0.020060
6,compiled_ratio,language,0.017704
7,active_release_score,release,0.017372
8,has_discussions,governance,0.015735
9,event_type_entropy,event,0.014724


## Auxiliary Experiment: Health Score Pseudo-label

이 실험은 `new_label`을 target으로 사용한다. 이 결과는 실제 외부 정답에 대한 성능이 아니라, feature 기반 health score 체계의 내부 일관성을 확인하는 보조 실험이다.

In [16]:
target_col = "new_label"

new_label_univariate = univariate_analysis(modeling_df, target_col)
new_label_linear_result = linear_odds_ratio(modeling_df, target_col)
new_label_nonlinear_result = nonlinear_importance(modeling_df, target_col)
new_label_ablation_result = ablation_test(modeling_df, target_col)

new_label_results, new_label_reports, new_label_models, new_label_split = evaluate_models(
    modeling_df,
    target_col,
)

print()
print("=" * 70)
print("FINAL MODEL PERFORMANCE: health score pseudo-label")
print("=" * 70)

new_label_results


X_train: (328, 51)
X_test : (83, 51)
y_train distribution:
new_label
0    0.5
1    0.5
Name: proportion, dtype: float64

LogisticRegression
              precision    recall  f1-score   support

           0       0.97      0.76      0.85        41
           1       0.80      0.98      0.88        42

    accuracy                           0.87        83
   macro avg       0.89      0.87      0.87        83
weighted avg       0.89      0.87      0.87        83

Confusion Matrix
[[31 10]
 [ 1 41]]

RandomForest
              precision    recall  f1-score   support

           0       0.93      0.93      0.93        41
           1       0.93      0.93      0.93        42

    accuracy                           0.93        83
   macro avg       0.93      0.93      0.93        83
weighted avg       0.93      0.93      0.93        83

Confusion Matrix
[[38  3]
 [ 3 39]]

ExtraTrees
              precision    recall  f1-score   support

           0       0.92      0.88      0.90        41

,target,model,accuracy,precision,recall,f1,roc_auc
0,new_label,GradientBoosting,0.891566,0.923077,0.857143,0.888889,0.968060
1,new_label,RandomForest,0.927711,0.928571,0.928571,0.928571,0.957027
2,new_label,ExtraTrees,0.903614,0.886364,0.928571,0.906977,0.956446
3,new_label,LogisticRegression,0.867470,0.803922,0.976190,0.881720,0.944832


In [17]:
new_label_linear_result.head(30)


,target,feature,family,coef,odds_ratio_per_1sd,abs_coef
0,new_label,release_recency_score,release,-1.556202,0.210936,1.556202
1,new_label,adoption_efficiency,popularity_adjusted,1.345297,3.839325,1.345297
2,new_label,total_contributions,contributor,1.325046,3.762360,1.325046
3,new_label,latest_tag_is_stable,release,1.322284,3.751981,1.322284
4,new_label,external_interest_event_ratio,event,1.312614,3.715874,1.312614
5,new_label,contribution_entropy,contributor,1.181160,3.258153,1.181160
6,new_label,activity_diversity_score,event,1.074753,2.929270,1.074753
7,new_label,repo_age_days,maintenance,0.950100,2.585967,0.950100
8,new_label,governance_openness_score,governance,0.887168,2.428244,0.887168
9,new_label,language_entropy,language,-0.852863,0.426193,0.852863


In [ ]:
new_label_nonlinear_result.head(30)

,target,feature,family,importance
0,new_label,has_IssueCommentEvent,event,0.084730
1,new_label,is_recently_pushed_30d,maintenance,0.063588
2,new_label,push_update_consistency,maintenance,0.056516
3,new_label,has_PullRequestEvent,event,0.041219
4,new_label,distributed_contribution_score,contributor,0.039028
5,new_label,maintainer_activity_score,maintenance,0.038292
6,new_label,contribution_entropy,contributor,0.037724
7,new_label,top1_contribution_share,contributor,0.031981
8,new_label,bus_factor_risk,contributor,0.030713
9,new_label,activity_diversity_score,event,0.029752


In [19]:
new_label_ablation_result


,target,setting,roc_auc_mean,roc_auc_std,num_features
0,new_label,without_event,0.970735,0.007716,38
1,new_label,without_governance,0.969746,0.018298,45
2,new_label,without_language,0.969250,0.019026,49
3,new_label,without_popularity_adjusted,0.968545,0.016490,46
4,new_label,all_features,0.967834,0.018146,51
5,new_label,without_maintenance,0.966636,0.020050,43
6,new_label,without_release,0.956528,0.022445,42
7,new_label,without_contributor,0.943778,0.034983,43
8,new_label,only_event,0.902966,0.052747,13
9,new_label,only_contributor,0.898626,0.027288,8


## 결과 저장

In [20]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

modeling_df.to_csv(OUTPUT_DIR / "oss_health_final_modeling_dataset.csv", index=False)
feature_selection_table.to_csv(OUTPUT_DIR / "oss_health_selected_features.csv", index=False)
rejected_feature_table.to_csv(OUTPUT_DIR / "oss_health_rejected_features.csv", index=False)

label_univariate.to_csv(OUTPUT_DIR / "label_univariate_analysis.csv", index=False)
label_linear_result.to_csv(OUTPUT_DIR / "label_linear_or_analysis.csv", index=False)
label_nonlinear_result.to_csv(OUTPUT_DIR / "label_nonlinear_importance.csv", index=False)
label_ablation_result.to_csv(OUTPUT_DIR / "label_ablation_result.csv", index=False)
label_results.to_csv(OUTPUT_DIR / "label_model_performance.csv", index=False)

new_label_univariate.to_csv(OUTPUT_DIR / "new_label_univariate_analysis.csv", index=False)
new_label_linear_result.to_csv(OUTPUT_DIR / "new_label_linear_or_analysis.csv", index=False)
new_label_nonlinear_result.to_csv(OUTPUT_DIR / "new_label_nonlinear_importance.csv", index=False)
new_label_ablation_result.to_csv(OUTPUT_DIR / "new_label_ablation_result.csv", index=False)
new_label_results.to_csv(OUTPUT_DIR / "new_label_model_performance.csv", index=False)

if not shap_importance.empty:
    shap_importance.to_csv(OUTPUT_DIR / "label_shap_importance.csv", index=False)

print("Saved outputs to:", OUTPUT_DIR.resolve())


Saved outputs to: /Users/carolyn/Desktop/3-1/opensource/data/oss-health-data/src/outputs
